In [1]:
from fastapi import FastAPI
from pydantic import BaseModel
import numpy as np
import joblib

In [2]:
app = FastAPI(title="Customer Churn Prediction API")

In [4]:
# load
model = joblib.load("model/model.pkl")
scaler = joblib.load("model/scaler.pkl")
columns = joblib.load("model/columns.pkl")

In [5]:
class Customer(BaseModel):
    tenure: float
    MonthlyCharges: float
    TotalCharges: float
    gender: int
    SeniorCitizen: int
    Partner: int
    Dependents: int
    PhoneService: int
    PaperlessBilling: int

In [7]:
@app.get("/")
def home():
    return {"message": "Churn API is running"}


In [9]:
@app.post("/predict")
def predict(data: Customer):
    try:
        input_dict = data.dict()

        input_df = pd.DataFrame([{}])
        input_df = input_df.reindex(columns=columns, fill_value=0)

        # Numeric features
        numeric_fields = ["tenure", "MonthlyCharges", "TotalCharges"]
        for field in numeric_fields:
            input_df[field] = input_dict[field]

        # Categorical encoding 
        for col in columns:
            if "_" in col:
                base, value = col.split("_", 1)
                if input_dict.get(base) == value:
                    input_df[col] = 1

        # Scale features
        input_scaled = scaler.transform(input_df)

        # Predict
        prediction = int(model.predict(input_scaled)[0])
        probability = float(model.predict_proba(input_scaled)[0][1])

        result = "Will Churn ❌" if prediction == 1 else "Will Stay ✅"

        return {
            "churn_prediction": prediction,
            "result": result,
            "probability": round(probability, 4)
        }

    except Exception as e:
        return {"error": str(e)}